# 01 — Quickstart

Runs the DMET pipeline on LiH end to end and checks the numbers against
validated references.

LiH takes seconds. Start here before anything larger.

**Prerequisite:** `pip install -e ".[qiskit]"` (or `--no-deps` if you have
your own PySCF build). The GQE solver is not needed for this notebook.

## 1. Verify the install first

`quenais-selftest` runs this whole pipeline and compares each quantity
against known-good values. If it fails, stop here — later numbers will not
mean anything.

In [ ]:
!quenais-selftest

## 2. Build a Config

Settings are grouped rather than flat. Molecule identity, paths and the
solver choice live on `Config`; everything else sits in a settings group.

In [ ]:
from quenais import Config
from quenais.settings import AsfSettings, DmetSettings

cfg = Config(
    molecule="LiH",
    basis="sto-3g",
    project_dir="./lih_run",
    classical_methods=["HF", "MP2", "CCSD", "CCSD_T"],
    dmet=DmetSettings(reference="casci"),
)
cfg.validate().make_dirs().load_geometry()

print(cfg)
print("geometry :", cfg.geometry)
print("bath tol :", cfg.dmet.bath_tolerance)
print("max embed:", cfg.dmet.max_embed_orbs)

## 3. Step 0 — classical references

These are the answer key. Every bug found during development was caught by
a disagreement with a number produced here.

Note the `reproducibility` column: CASSCF and NEVPT2 are optimiser-dependent
and will not reproduce to tight tolerance across machines. See
`docs/limitations.md`.

In [ ]:
from quenais.classical import runner

step0 = runner.main(cfg, force=True)

## 4. Step 1 — active space

ASF selects orbitals by entanglement entropy, then a degeneracy-aware gap
cutoff narrows the list. The cutoff is extended rather than allowed to
split a degenerate pair — splitting one breaks the molecule's symmetry.

In [ ]:
from quenais.active_space import finder

step1 = finder.main(cfg, force=True)
print()
print("active space :", f"({step1['nel']}e, {step1['n_active_orbs']}o)")
print("MOs          :", step1["mo_list"])

## 5. Step 2 — the DMET embedding

Watch two diagnostics in the output:

- **the Schmidt singular values.** These decide the bath. If none clears
  `bath_tolerance`, the correct answer is zero bath orbitals — not a
  fabricated bath from numerical noise.
- **the embedded electron count.** It comes from the reference density,
  not the active-space count. LiH's active space holds 2 electrons but its
  embedding space holds 4; using the active-space count roughly doubles
  the energy.

In [ ]:
from quenais.embedding import hamiltonian

step2 = hamiltonian.main(cfg, force=True)

## 6. The check that matters

`embedded_scf_check` runs a real, converged SCF on the embedding
Hamiltonian and compares it with the full-molecule UHF energy.

This is the single most diagnostic quantity in the pipeline. The `ecore`
self-consistency identity printed by some DMET codes is tautological —
`ecore` is *defined* as that difference, so it can never fail. This one
can.

In [ ]:
check = step2["embedded_scf_check"]
for k, v in check.items():
    print(f"  {k:15} {v}")

## 7. Compare against the validated value

LiH's DMET+CASCI reference is **−7.881246152 Ha**.

In [ ]:
from quenais.visualization.plots import true_embedding_casci

e_casci = true_embedding_casci(cfg)
reference = -7.881246152

print(f"DMET+CASCI : {e_casci:.9f} Ha")
print(f"reference  : {reference:.9f} Ha")
print(f"difference : {abs(e_casci - reference) * 1e3:.4f} mHa")

## 8. Step 4 — figures and summary

Produces whatever the available data supports. With no GQE log the GQE
figures are skipped rather than drawn empty.

In [ ]:
from quenais.visualization import plots

result = plots.main(cfg)
print()
print(open(result["results_summary"]).read())

## Same thing from the command line

```bash
quenais-run --molecule LiH --basis sto-3g --steps 0 1 2 4 --project-dir ./lih_run
```